In [ ]:
import json
import os
import torch
CLASSES = {'go': 0, 'java': 1, 'javascript': 2, 'php': 3, 'python': 4, 'ruby': 5}
INV_CLASSES = ['go', 'java', 'javascript', 'php', 'python', 'ruby']
CONCEPTS= ["comments", "function_declarations", "go_function_declarations", "java_function_declarations", "javascript_function_declarations", "php_function_declarations","python_function_declarations", "ruby_function_declarations"]
CLASSES_TO_EXAMINE = ['go', 'java', 'javascript', 'php', 'python', 'ruby']
MODEL_NAME = "huggingface/CodeBERTa-language-id"

data = []
with open(f"./data/code_classification/random_concept_dataset.jsonl", "r") as f:
    rnd = [json.loads(line) for line in f]
    for x in rnd:
        data.append({'text': x['text'], 'label': CLASSES[x['language']]})
        
TEST = ([d["text"] for d in data], [d["label"] for d in data])

from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES_TO_EXAMINE))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
MAIN_DATASETS = {}
def load_dataset(name):
    with open(f"./datasets/codesearch_data/{name}.jsonl", "r") as f:
        return [json.loads(line) for line in f]
    

MAIN_DATASETS["complete"] = load_dataset("all_codes")
MAIN_DATASETS["comments"] = load_dataset("all_comments")
MAIN_DATASETS["functions"] = load_dataset("all_functions")
MAIN_DATASETS["go_functions"] = load_dataset("go_functions")
MAIN_DATASETS["java_functions"] = load_dataset("java_functions")
MAIN_DATASETS["javascript_functions"] = load_dataset("javascript_functions")
MAIN_DATASETS["php_functions"] = load_dataset("php_functions")
MAIN_DATASETS["python_functions"] = load_dataset("python_functions")
MAIN_DATASETS["ruby_functions"] = load_dataset("ruby_functions")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from sklearn.linear_model import SGDClassifier
import random
import string

class TCAV:
    """ Class for concept activation vectors for Keras models.

    Attributes:
        model: a roBerta LLM loaded trough huggingface API
        tokenizer: a tokenizer for roBerta
        cav: A numpy array containing the concept activation vector
        sensitivity: A numpy array containing sensitivities
        y_labels: A numpy array containing class labels
        bottleneck: a int indicating to which hidden layer extract the activations
    """

    def __init__(self, model=None, tokenizer=None, fix_length=None):
        self.model = model
        self.tokenizer = tokenizer
        self.cavs = {}  # dict: concept_name -> cav
        self.sensitivities = {}  # dict: concept_name -> sensitivities
        self.y_labels = {}  # dict: concept_name -> y_labels
        self.current_concept = None
        self.bottleneck = [] # list of bottleneck layers to analyze
        self.model_activations = {}
        self.fix_length = fix_length
    
    def forward_hook_fn(self, name:str):
        if name not in self.model_activations.keys():
            self.model_activations["forward_"+name] = []
        def fn(module, input, output):
            # print("leaf",output.is_leaf)
            # x = output
            # print("leaf",output[0].is_leaf)
            x = output[0]
            x.requires_grad_(True)
            x.retain_grad()
            self.model_activations["forward_"+name].append(x)
            # print("extracted activations", output.shape)
            # print("extracted activations", output[0].shape)
            # print("----------------",output[0].shape[1],"---------------")
            # self.fix_length = output[0].shape[1]
            return
        return fn
    
    def backward_hook_fn(self, name:str):
        if name not in self.model_activations.keys():
            self.model_activations["backward_"+name] = []
            
        def fn(module, grad_input, grad_output):
            # print("leaf",grad_output[0].is_leaf)
            
            self.model_activations["backward_"+name] = grad_output[0]
            # print("extracted grads", grad_output[0].shape)
            return
        
        return fn
    
    def set_concept(self, concept):
        """ Set the concept name """
        self.current_concept = concept

    def set_model(self, model):
        """ Set the model """
        self.model = model
        return

    def set_tokenizer(self, tokenizer):
        """ Set the tokenizer """
        self.tokenizer = tokenizer
        return

    def split_model(self, bottleneck):
        """ Set the hook at the bottleneck layer """
        if bottleneck < 0 or bottleneck >= len(self.model.roberta.encoder.layer):
            raise ValueError("Invalid layer for sampling")
        
        # layers = list(self.model.children())
        # print(layers)
        self.bottleneck.append(str(bottleneck))   
        
        # self.model.classifier.dense.register_forward_hook(self.hook_fn(str(bottleneck)))
        self.model.roberta.encoder.layer[bottleneck].register_forward_hook(self.forward_hook_fn(str(bottleneck)))
        self.model.roberta.encoder.layer[bottleneck].register_full_backward_hook(self.backward_hook_fn(str(bottleneck)))
        return


    def _create_counterexamples(self, x_concept):
        """ Creates random counterexamples to a series of concept inputs """
        n = len(x_concept)
    
        counterexamples = []
        for i in range(n):
            l = len(x_concept[i])
            counterexamples.append(''.join(random.choices(string.printable, k=l)))
        return counterexamples

    def _tokenize(self, inputs):
        """ Tokenize the inputs (if tokenizer is provided) """
        # print(len(inputs))
        if self.tokenizer is not None:
            if self.fix_length:
                x = self.tokenizer(inputs, return_tensors="pt", padding="max_length", max_length=self.fix_length, truncation=True)
            else:
                x = self.tokenizer(inputs, return_tensors="pt", padding="longest", truncation=True)
                self.fix_length = x["input_ids"].shape[1]
            # print("tokenizer", x["input_ids"].shape)
            return x
        return inputs

    def train_cav(self, x_concept):
        """ Train and extract the Concept Activation Vector """
        
        counterexamples = self._create_counterexamples(x_concept)
        tmp = x_concept + counterexamples
        x_train_concept = self._tokenize(tmp)
        y_train_concept = torch.cat((torch.ones(len(x_concept)), torch.zeros(len(counterexamples))))
        
        print("calculating cavs")
        # Obtain activations of concept and counterexamples
        with torch.no_grad():
            _ = self.model(**x_train_concept)
            # print("attentions dimensions:", len(self.model_activations["forward_"+self.bottleneck][0].shape))
            if(len(self.model_activations["forward_"+self.bottleneck][0].shape)>2):
                # Flatten the activations if they are not linear: (n, m, z) to (n, m*z)
                concept_activations = {}
                for bottleneck in self.bottleneck:
                    concept_activations[bottleneck] = self.model_activations["forward_"+bottleneck][0].reshape(self.model_activations["forward_"+bottleneck][0].shape[0],-1)
            else:
                # If activations are linear, no flattening need: (n, m)
                concept_activations = {}
                for bottleneck in self.bottleneck:
                    concept_activations[bottleneck] = self.model_activations["forward_"+bottleneck][0]
            # print("concept activations shape", concept_activations.shape)
        # print(concept_activations.shape)

        # Iterate over all bottlenecks
        # print("bottlenecks", self.bottleneck)
        self.cavs[self.current_concept] = {}

        for b in concept_activations.keys():
            # Train linear classifier
            lm = SGDClassifier(loss="perceptron", eta0=1, learning_rate="constant", penalty=None)
            lm.fit(concept_activations[b].detach().numpy(), y_train_concept.numpy())
            cav = -lm.coef_.T
            
            self.cavs[self.current_concept][b] = cav
            # print("cav", len(cav))
        
            self.model_activations["forward_"+b] = [] #once calculated all results, reset for next operations                           
            self.model_activations["backward_"+b] = []
        return
        
    def calculate_sensitivity(self, x_train, y_train, device="cpu"):
        """
        This function calculates the TCAV sensitivity scores for a given set of inputs and labels.
        It computes the gradient of the loss with respect to the activations at the bottleneck layer,
        projects these gradients onto the concept activation vector (CAV), and measures how sensitive
        the model's predictions are to the concept for each class. The results are stored for later analysis.
        """
        
        print("calculating sensitivity")
        x_train = self._tokenize(x_train)
        # print(x_train)
        
        # Calculate the output and obtain activations
        x_train = x_train.to(device)
        output = self.model(**x_train)
        
        #activations = self.model_activations["forward_"+self.bottleneck][0].reshape(self.model_activations["forward_"+self.bottleneck][0].shape[0],-1) # Prendi l'ultima attivazione del bottleneck
        # print("output logits", output.logits.shape)
        # print("activation shape", activations.shape)
        
        # Control the format
        if isinstance(y_train, list):
            y_train = np.array(y_train)
            # print(y_train)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.from_numpy(y_train)
        y_labels = y_train.view(-1).to(device)
        # print(y_labels)

        # Define and compute the loss
        loss = F.cross_entropy(output.logits, y_labels)
        # print("loss", loss)
        # print("activations",activations.is_leaf, activations)
        # print("activations requires grad", activations.requires_grad)

        # Calculate the gradient
        loss.backward()
        
        grads = {}
        for bottleneck in self.bottleneck:
            grads[bottleneck] = self.model_activations["backward_"+bottleneck]
    
        # grads = torch.autograd.grad(loss, activations, allow_unused=True)
        # print("grads", grads)
        # concatenate grads
        grads = {k: v.reshape(v.shape[0], -1) for k, v in grads.items()}

        # Scalar product
        cavs = self.cav
        
        # print("shapes", cavs.shape, grads.shape)
        sensitivities = {}
        for b in cavs.keys():
            sensitivities[b] = []
            cav_tensor = cavs[b]
            # print("cav_tensor", cav_tensor.shape)
            for g in grads[b]:
                sensitivities.append(np.dot(g, cav_tensor))
        
        
            sensitivities[b] = np.array(sensitivities[b])
        # print("sensitivity", sensitivity)

        # Saving sensitivity
        self.sensitivities[self.current_concept] = sensitivities
        self.y_labels[self.current_concept] = y_train.detach().cpu().numpy().reshape(-1)

        return
        
    def print_all_sensitivities(self, id_to_labels):
        for concept, sensitivities_dict in self.sensitivities.items():
            y_labels_dict = self.y_labels[concept]
            print(f"Sensitività per il concetto '{concept}':")
            for bottleneck, sensitivity in sensitivities_dict.items():
                y_labels = y_labels_dict  # y_labels are the same for all bottlenecks
                print(f"  Bottleneck {bottleneck}:")
                num_labels = len(np.unique(y_labels))
                for label_idx in range(num_labels):
                    idxs = np.where(y_labels == label_idx)[0]
                    value = np.sum(sensitivity[idxs] > 0) / idxs.shape[0]
                    print(f"    Classe {id_to_labels[label_idx]}: {value:.2f}")
                print("-" * 40)

    def get_sensitivity_results(self, id_to_labels):
        """
        Returns a list of dicts with the sensitivity results for all concepts, bottlenecks and classes.
        Format:
        [
            {"concept": "...", "bottleneck": "...", "class": "...", "sensitivity": 0.xx},
            ...
        ]
        """
        results = []
        for concept, sensitivities_dict in self.sensitivities.items():
            y_labels_dict = self.y_labels[concept]
            for bottleneck, sensitivity in sensitivities_dict.items():
                y_labels = y_labels_dict
                num_labels = len(np.unique(y_labels))
                for label_idx in range(num_labels):
                    idxs = np.where(y_labels == label_idx)[0]
                    value = np.sum(sensitivity[idxs] > 0) / idxs.shape[0]
                    results.append({
                        "concept": concept,
                        "bottleneck": bottleneck,
                        "class": id_to_labels[label_idx],
                        "sensitivity": value
                    })
        return results
    


class TCAV_Avg:
    """ Class for concept activation vectors for Keras models.

    Attributes:
        model: a roBerta LLM loaded trough huggingface API
        tokenizer: a tokenizer for roBerta
        cav: A numpy array containing the concept activation vector
        sensitivity: A numpy array containing sensitivities
        y_labels: A numpy array containing class labels
        bottleneck: a int indicating to which hidden layer extract the activations
    """

    def __init__(self, model=None, tokenizer=None, fix_length=None):
        self.model = model
        self.tokenizer = tokenizer
        self.cavs = {}  # dict: concept_name -> cav
        self.sensitivities = {}  # dict: concept_name -> sensitivities
        self.y_labels = {}  # dict: concept_name -> y_labels
        self.current_concept = None
        self.bottleneck = None
        self.model_activations = {}
        self.fix_length = fix_length
    
    def forward_hook_fn(self, name:str):
        if name not in self.model_activations.keys():
            self.model_activations["forward_"+name] = []
        def fn(module, input, output):
            # print("leaf",output.is_leaf)
            # x = output
            # print("leaf",output[0].is_leaf)
            x = output[0]
            x.requires_grad_(True)
            x.retain_grad()
            self.model_activations["forward_"+name].append(x)
            # print("extracted activations", output.shape)
            # print("extracted activations", output[0].shape)
            # print("----------------",output[0].shape[1],"---------------")
            # self.fix_length = output[0].shape[1]
            return
        return fn
    
    def backward_hook_fn(self, name:str):
        if name not in self.model_activations.keys():
            self.model_activations["backward_"+name] = []
            
        def fn(module, grad_input, grad_output):
            # print("leaf",grad_output[0].is_leaf)
            
            self.model_activations["backward_"+name] = grad_output[0]
            # print("extracted grads", grad_output[0].shape)
            return
        
        return fn

    def set_concept(self, concept):
        """ Set the concept name """
        self.current_concept = concept
        
    def set_model(self, model):
        """ Set the model """
        self.model = model
        return

    def set_tokenizer(self, tokenizer):
        """ Set the tokenizer """
        self.tokenizer = tokenizer
        return

    def split_model(self, bottleneck):
        """ Set the hook at the bottleneck layer """
        if bottleneck < 0 or bottleneck >= len(self.model.roberta.encoder.layer):
            raise ValueError("Invalid layer for sampling")
        
        # layers = list(self.model.children())
        # print(layers)
        self.bottleneck = str(bottleneck)   
        
        # self.model.classifier.dense.register_forward_hook(self.hook_fn(str(bottleneck)))
        self.model.roberta.encoder.layer[bottleneck].register_forward_hook(self.forward_hook_fn(str(bottleneck)))
        self.model.roberta.encoder.layer[bottleneck].register_full_backward_hook(self.backward_hook_fn(str(bottleneck)))
        return


    def _create_counterexamples(self, x_concept):
        """ Creates random counterexamples to a series of concept inputs """
        n = len(x_concept)
    
        counterexamples = []
        for i in range(n):
            l = len(x_concept[i])
            counterexamples.append(''.join(random.choices(string.printable, k=l)))
        return counterexamples

    def _tokenize(self, inputs):
        """ Tokenize the inputs (if tokenizer is provided) """
        # print(len(inputs))
        if self.tokenizer is not None:
            x = self.tokenizer(inputs, return_tensors="pt", padding=True, truncation=True)
            # print("tokenizer", x["input_ids"].shape)
            return x
        return inputs

    def train_cav(self, x_concept):
        """ Train and extract the Concept Activation Vector """
        
        counterexamples = self._create_counterexamples(x_concept)
        tmp = x_concept + counterexamples
        x_train_concept = self._tokenize(tmp)
        y_train_concept = torch.cat((torch.ones(len(x_concept)), torch.zeros(len(counterexamples))))
        
        print("calculating cavs")
        # Obtain activations of concept and counterexamples
        with torch.no_grad():
            _ = self.model(**x_train_concept)
            # print("attentions dimensions:", len(self.model_activations["forward_"+self.bottleneck][0].shape))
            if(len(self.model_activations["forward_"+self.bottleneck][0].shape)>2):
                # Instead of flattening (n, m, z) to (n, m*z), take the mean over m to get (n, z)
                # Calcola la media solo sui token non di padding usando l'attenzione mask
                activations = self.model_activations["forward_"+self.bottleneck][0]  # (batch, seq_len, hidden)
                attention_mask = x_train_concept["attention_mask"]  # (batch, seq_len)
                mask = attention_mask.unsqueeze(-1).expand(activations.size())  # (batch, seq_len, hidden)
                activations_masked = activations * mask  # maschera i padding
                lengths = attention_mask.sum(dim=1).unsqueeze(-1)  # (batch, 1)
                # Evita divisione per zero
                lengths = lengths.clamp(min=1)
                concept_activations = activations_masked.sum(dim=1) / lengths
            else:
                concept_activations = self.model_activations["forward_"+self.bottleneck][0]
            # print("concept activations shape", concept_activations.shape)
        # print(concept_activations.shape)
        ### Concatenate token by token
        
        # Train linear classifier
        lm = SGDClassifier(loss="perceptron", eta0=1, learning_rate="constant", penalty=None)
        lm.fit(concept_activations.detach().numpy(), y_train_concept.numpy())
        cav = -lm.coef_.T
        self.cavs[self.current_concept] = cav
        # print("cav", len(cav))
        
        self.model_activations["forward_"+self.bottleneck] = [] #once calculated all results, reset for next operations                           
        self.model_activations["backward_"+self.bottleneck] = []
        return
        
    def calculate_sensitivity(self, x_train, y_train, device="cpu"):
        """
        Versione PyTorch della funzione, con commenti che rimandano
        ai passaggi originali in Keras.
        """
        
        print("calculating sensitivity")
        x_train = self._tokenize(x_train)
        # print(x_train)
        
        # Calculate the output and obtain activations
        x_train = x_train.to(device)
        output = self.model(**x_train)

        # Take the mean over the sequence dimension (m) to get (n, z)
        # activations = self.model_activations["forward_"+self.bottleneck][0].mean(dim=1)
        # print("output logits", output.logits.shape)
        # print("activation shape", activations.shape)
        
        # Control the format
        if isinstance(y_train, list):
            y_train = np.array(y_train)
            # print(y_train)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.from_numpy(y_train)
        y_labels = y_train.view(-1).to(device)
        # print(y_labels)

        # Define and compute the loss
        loss = F.cross_entropy(output.logits, y_labels)
        # print("loss", loss)
        # print("activations",activations.is_leaf, activations)
        # print("activations requires grad", activations.requires_grad)

        # Calculate the gradient
        loss.backward()
        
        grads = self.model_activations["backward_"+self.bottleneck]
        # grads = torch.autograd.grad(loss, activations, allow_unused=True)
        # print("grads", grads)
        # concatenate grads
        # Se grads ha più di 2 dimensioni, calcola la media solo sui token non di padding
        if grads.dim() > 2:
            attention_mask = x_train["attention_mask"]  # (batch, seq_len)
            mask = attention_mask.unsqueeze(-1).expand(grads.size())  # (batch, seq_len, hidden)
            grads_masked = grads * mask  # maschera i padding
            lengths = attention_mask.sum(dim=1).unsqueeze(-1)  # (batch, 1)
            lengths = lengths.clamp(min=1)
            grads = grads_masked.sum(dim=1) / lengths

        # Scalar product
        cav_tensor = self.cav
        
        # print("shapes", cav_tensor.shape, grads.shape)
        sensitivities = []
        for g in grads:
            sensitivities.append(np.dot(g, cav_tensor))
        
        sensitivity = np.array(sensitivities)
        # print("sensitivity", sensitivity)

        # Saving sensitivity
        self.sensitivities[self.current_concept] = sensitivity
        self.y_labels[self.current_concept] = y_train.detach().cpu().numpy().reshape(-1)

        return
        
def print_all_sensitivities(self, id_to_labels):
        for concept, sensitivity in self.sensitivities.items():
            y_labels = self.y_labels[concept]
            print(f"Sensitività per il concetto '{concept}':")
            num_labels = len(np.unique(y_labels))
            for label_idx in range(num_labels):
                value = np.sum(sensitivity[np.where(y_labels == label_idx)[0]] > 0) / np.where(y_labels == label_idx)[0].shape[0]
                print(f"  Classe {id_to_labels[label_idx]}: {value:.2f}")
            print("-" * 40)
            
def get_sensitivity_results(self, id_to_labels):
    """
    Returns a list of dicts with the sensitivity results for all concepts and classes.
    Format:
    [
        {"concept": "...", "class": "...", "sensitivity": 0.xx},
        {"concept": "...", "class": "...", "sensitivity": 0.xx},
        ...
    ]
    """
    results = []
    for concept, sensitivity in self.sensitivities.items():
        y_labels = self.y_labels[concept]
        num_labels = len(np.unique(y_labels))
        for label_idx in range(num_labels):
            value = np.sum(sensitivity[np.where(y_labels == label_idx)[0]] > 0) / np.where(y_labels == label_idx)[0].shape[0]
            results.append({
                "concept": concept,
                "class": id_to_labels[label_idx],
                "sensitivity": value
            })
    return results

In [ ]:
for run in range(10):
    # Initialize TCAV instances for different configurations
    tcav_auto = TCAV(model=model, tokenizer=tokenizer)
    tcav_fixed = TCAV(model=model, tokenizer=tokenizer, fix_length=512)
    tcav_avg = TCAV_Avg(model=model, tokenizer=tokenizer)
    
    # Random extract samples from main datasets
    samples = {}
    for dataset_name, dataset in MAIN_DATASETS.items():
        samples[dataset_name] = random.sample(dataset, 1000)
        
    # Set the bottleneck layer
    
    

